In [1]:
!pip install langchain langchain-core langchain-community pydantic duckduckgo-search langchain_experimental

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 3.6 MB/s  0:00:01m 3.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 8.5 MB/s  0:00:008.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [langchain_experimental]4 [langchain_experimental]


#Built-in Tool - DuckDuckGo Search

In [3]:
!pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 5.8 MB/s  0:00:00m 7.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [ddgs]m━━━━━ 6/7 [ddgs]


In [5]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

results = search_tool.invoke('top news about psl')

print(results)

4 days ago - The PSL traditionally been held between February and March each year, with the only major disruption occurring during the COVID-19 pandemic . The league previously took place before the start of the Indian Premier League. The 2025 season was the last to be held in the traditional February-March ... 2 days ago · All the Latest News from the PSL . Team and player profiles, natch coverage, schedules and results. May 13, 2025 · Get the latest authentic and unbiased PSL News in Pakistan: breaking news , scoops, features, exclusives and analysis. PSL Latest News Updates - PSL 2025 News , PSL 10 Updates, Breaking News - Geo Super. Read latest Pakistan Super League News , Headlines of today and archives of news . Latest and updated Breaking news including headlines, current affairs, analysis, and indepth stories.


In [6]:
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

duckduckgo_search
A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
{'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


In [7]:
#Built-in Tool - Shell Tool

##Built-in Tool - Shell Tool

In [8]:
from langchain_community.tools import ShellTool

shell_tool = ShellTool()

results = shell_tool.invoke('ls')

print(results)

Executing command:
 ls
buildin.ipynb



/home/talha/code/langchain projects/.venv/lib/python3.13/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


In [9]:
from langchain_core.tools import tool


# Step 1 - create a function

def multiply(a, b):
    """Multiply two numbers"""
    return a*b

In [10]:
# Step 2 - add type hints

def multiply(a: int, b:int) -> int:
    """Multiply two numbers"""
    return a*b

In [11]:
# Step 3 - add tool decorator

@tool
def multiply(a: int, b:int) -> int:
    """Multiply two numbers"""
    return a*b

In [14]:
result = multiply.invoke({"a":9, "b":5})

In [15]:
print(result)

45


In [16]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [17]:
print(multiply.args_schema.model_json_schema())

{'description': 'Multiply two numbers', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


In [ ]:
from langchain_core.tools import tool
from pydantic import BaseModel, Field

class MultiplyInput(BaseModel):
    a: int = Field(..., description="The first number")
    b: int = Field(..., description="The second number")

@tool(args_schema=MultiplyInput)
def multiply_func(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b


In [25]:
result = multiply_func.invoke({"a": 5, "b": 6})
print(result)   # 30


print(result)
print(multiply_func.name)
print(multiply_func.description)
print(multiply_func.args)

30
30
multiply_func
Multiply two numbers.
{'a': {'description': 'The first number', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number', 'title': 'B', 'type': 'integer'}}


Method 3 - Using BaseTool Class

In [26]:
from langchain.tools import BaseTool
from typing import Type

In [27]:
# arg schema using pydantic

class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

/tmp/ipykernel_347905/908171234.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  a: int = Field(required=True, description="The first number to add")
/tmp/ipykernel_347905/908171234.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  b: int = Field(required=True, description="The second number to add")


In [28]:
class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"

    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        return a * b

In [29]:
multiply_tool = MultiplyTool()

In [30]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)

print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'required': True, 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'required': True, 'title': 'B', 'type': 'integer'}}


## Toolkit


In [31]:
from langchain_core.tools import tool

# Custom tools
@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b


In [32]:
class MathToolkit:
    def get_tools(self):
        return [add, multiply]


In [33]:

toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, "=>", tool.description)


add => Add two numbers
multiply => Multiply two numbers
